# Model 3 — 09 Controlled Tuning Experiment

## Controlled tuning experiment
Train a small, reproducible candidate set on the existing processed training cache. This creates an **experiment artifact only**. The Friday holdout is not used for tuning.

In [1]:

from pathlib import Path
import json, warnings, time
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_recall_fscore_support, roc_auc_score, average_precision_score
from sklearn.model_selection import train_test_split

PROJECT_ROOT=Path.cwd()
while PROJECT_ROOT.name!="EVIDENT" and PROJECT_ROOT.parent!=PROJECT_ROOT:
    PROJECT_ROOT=PROJECT_ROOT.parent
DATA_DIR=PROJECT_ROOT/"data"/"raw"/"behaviour"
MODEL_DIR=PROJECT_ROOT/"models"/"behavior"
META_DIR=MODEL_DIR/"metadata"
PROD_DIR=MODEL_DIR/"production"
META_DIR.mkdir(parents=True,exist_ok=True)

cache=PROJECT_ROOT/"data"/"processed"/"behavior"/"cicids2017_model3_training.parquet"
schema=json.loads((META_DIR/"feature_schema.json").read_text())
df=pd.read_parquet(cache); features=schema["feature_columns"]
X=df[features]; yb=df["is_attack"]; yf=df["attack_family"]
Xtr,Xv,ytr,yv,ftr,fv=train_test_split(X,yb,yf,test_size=.2,random_state=42,stratify=yb)

candidates=[
{"name":"baseline","max_iter":250,"learning_rate":.08,"max_leaf_nodes":31,"l2":1.0},
{"name":"regularized","max_iter":350,"learning_rate":.05,"max_leaf_nodes":31,"l2":3.0},
{"name":"higher_capacity","max_iter":350,"learning_rate":.06,"max_leaf_nodes":63,"l2":1.0},
{"name":"conservative","max_iter":450,"learning_rate":.04,"max_leaf_nodes":31,"l2":5.0}]

def build(c):
    return Pipeline([("imputer",SimpleImputer(strategy="median")),("scaler",StandardScaler()),("classifier",HistGradientBoostingClassifier(max_iter=c["max_iter"],learning_rate=c["learning_rate"],max_leaf_nodes=c["max_leaf_nodes"],l2_regularization=c["l2"],random_state=42))])

results=[]; best=None; bestfam=None; bestcfg=None; bestscore=-1
for c in candidates:
    bm=build(c); bm.fit(Xtr,ytr)
    bp=bm.predict(Xv)
    bf=precision_recall_fscore_support(yv,bp,average="binary",zero_division=0)[2]
    fm=build(c); m=ftr!="BENIGN"; fm.fit(Xtr[m],ftr[m])
    am=fv!="BENIGN"; fp=fm.predict(Xv[am])
    ff=precision_recall_fscore_support(fv[am],fp,average="macro",zero_division=0)[2]
    score=.6*bf+.4*ff
    results.append({"name":c["name"],"binary_f1":float(bf),"family_macro_f1":float(ff),"combined_score":float(score)})
    if score>bestscore: bestscore=score; best=bm; bestfam=fm; bestcfg=c

res=pd.DataFrame(results).sort_values("combined_score",ascending=False)
print(res.to_string(index=False))
exp=MODEL_DIR/"experiments"; exp.mkdir(parents=True,exist_ok=True)
import joblib
joblib.dump({"artifact_type":"EVIDENT_MODEL3_EXPERIMENT","artifact_version":"1.1.0-experiment","config":bestcfg,"binary_model":best,"family_model":bestfam,"feature_columns":features},exp/"network_threat_model_experiment.pkl",compress=3)
res.to_json(exp/"tuning_results.json",orient="records",indent=2)
print("Experiment artifact:",exp/"network_threat_model_experiment.pkl")


           name  binary_f1  family_macro_f1  combined_score
higher_capacity   0.998998         0.898055        0.958621
   conservative   0.999011         0.880391        0.951563
    regularized   0.999011         0.877748        0.950506
       baseline   0.998959         0.877519        0.950383
Experiment artifact: c:\Users\bimal\OneDrive\Desktop\EVIDENT\models\behavior\experiments\network_threat_model_experiment.pkl
